# Step 5: Parallel Range + Azimuth Banks


Adds two parallel coincidence branches for separate range and azimuth classification from binaural returns.


In [ ]:

from pathlib import Path
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("DEVICE:", DEVICE)


def save_fig(plot_dir: Path, name: str):
    path = plot_dir / name
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved plot:", path)


def confusion_matrix_plot(y_true, y_pred, class_labels, plot_dir: Path, filename: str):
    n = len(class_labels)
    cm = np.zeros((n, n), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    im0 = axes[0].imshow(cm, cmap="Blues", aspect="auto")
    axes[0].set_title("Confusion (counts)")
    axes[0].set_xlabel("Pred")
    axes[0].set_ylabel("True")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0.0, vmax=1.0, aspect="auto")
    axes[1].set_title("Confusion (row-normalized)")
    axes[1].set_xlabel("Pred")
    axes[1].set_ylabel("True")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    tick_idx = np.arange(n)
    for ax in axes:
        ax.set_xticks(tick_idx)
        ax.set_yticks(tick_idx)
        ax.set_xticklabels(class_labels, rotation=90)
        ax.set_yticklabels(class_labels)

    plt.tight_layout()
    save_fig(plot_dir, filename)
    plt.show()

# Step 5: Two parallel coincidence banks (range + azimuth)
ART = Path("artifacts_steps/step5_parallel_range_azimuth")
PLOT_DIR = ART / "plots"
CKPT = ART / "model_step5.pt"
METRICS = ART / "metrics_step5.json"
for d in [ART, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DT = 1e-4
FS = 1.0 / DT
T_STEPS = 768
TIME = np.arange(T_STEPS) * DT
SPEED_OF_SOUND = 343.0

RANGE_MIN, RANGE_MAX = 0.5, 10.0
RANGE_CLASSES = 20
RANGE_BINS = np.linspace(RANGE_MIN, RANGE_MAX, RANGE_CLASSES + 1)
RANGE_CENTERS = 0.5 * (RANGE_BINS[:-1] + RANGE_BINS[1:])

AZ_MIN, AZ_MAX = -45.0, 45.0
AZ_CLASSES = 9
AZ_BINS = np.linspace(AZ_MIN, AZ_MAX, AZ_CLASSES + 1)
AZ_CENTERS = 0.5 * (AZ_BINS[:-1] + AZ_BINS[1:])

N_SAMPLES = 1400
BATCH = 64
EPOCHS = 20

ECHO_ATT = 0.55
DELAY_STEPS = torch.arange(0, 673, 24).long()
FREQ_CH = 8
HIDDEN = 128


def chirp_tx():
    t = TIME
    dur = 0.02
    f_hi, f_lo = 4000.0, 1200.0
    k = (f_lo - f_hi) / dur
    phase = 2 * np.pi * (f_hi * t + 0.5 * k * np.minimum(t, dur) ** 2)
    env = np.exp(-0.5 * ((np.arange(T_STEPS) - 70) / 22.0) ** 2)
    sig = np.sin(phase) * env
    sig[t > dur] *= 0.0
    return sig.astype(np.float32)


def range_bin(d):
    return int(np.digitize(d, RANGE_BINS[1:-1], right=False))


def az_bin(a):
    return int(np.digitize(a, AZ_BINS[1:-1], right=False))


def make_sample(distance_m, az_deg):
    tx = chirp_tx()
    delay_steps = int(round((2.0 * distance_m / SPEED_OF_SOUND) / DT))

    az_norm = np.clip(az_deg / 45.0, -1.0, 1.0)
    g_left = np.clip(1.0 - 0.35 * az_norm, 0.2, 1.8)
    g_right = np.clip(1.0 + 0.35 * az_norm, 0.2, 1.8)

    rx_l = np.zeros_like(tx)
    rx_r = np.zeros_like(tx)
    if delay_steps < T_STEPS:
        base = ECHO_ATT * tx[: T_STEPS - delay_steps]
        rx_l[delay_steps:] = g_left * base
        rx_r[delay_steps:] = g_right * base

    return tx.astype(np.float32), rx_l.astype(np.float32), rx_r.astype(np.float32)


def build_data(seed=SEED):
    rng = np.random.default_rng(seed)
    ranges = rng.uniform(RANGE_MIN, RANGE_MAX, size=N_SAMPLES)
    az = rng.uniform(AZ_MIN, AZ_MAX, size=N_SAMPLES)

    txs, rls, rrs, yr, ya = [], [], [], [], []
    for d, a in zip(ranges, az):
        tx, rl, rr = make_sample(float(d), float(a))
        txs.append(tx); rls.append(rl); rrs.append(rr)
        yr.append(range_bin(d)); ya.append(az_bin(a))

    txs = np.stack(txs); rls = np.stack(rls); rrs = np.stack(rrs)
    yr = np.array(yr, dtype=np.int64); ya = np.array(ya, dtype=np.int64)

    perm = rng.permutation(N_SAMPLES)
    txs, rls, rrs, yr, ya = txs[perm], rls[perm], rrs[perm], yr[perm], ya[perm]

    split = int(0.8 * N_SAMPLES)
    return txs[:split], rls[:split], rrs[:split], yr[:split], ya[:split], txs[split:], rls[split:], rrs[split:], yr[split:], ya[split:]


class FrequencyFilterBank(nn.Module):
    def __init__(self, num_channels=8, kernel_size=33, fs=FS, f_low=800.0, f_high=4200.0):
        super().__init__()
        self.conv = nn.Conv1d(1, num_channels, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        t = np.arange(kernel_size) / fs
        t = t - t.mean()
        win = np.hanning(kernel_size)
        freqs = np.linspace(f_low, f_high, num_channels)
        kernels = []
        for f in freqs:
            k = np.sin(2 * np.pi * f * t) * win
            k = k / (np.linalg.norm(k) + 1e-8)
            kernels.append(k.astype(np.float32))
        w = torch.from_numpy(np.stack(kernels)[:, None, :])
        with torch.no_grad():
            self.conv.weight.copy_(w)
        self.conv.weight.requires_grad = False

    def forward(self, x):
        xb = x.permute(1,2,0)
        y = self.conv(xb)
        return y.permute(2,0,1)


class MultiChannelDelayBank(nn.Module):
    def __init__(self, delays):
        super().__init__()
        self.register_buffer("delays", delays.clone().long())
    def forward(self, x):
        T,B,Fch = x.shape
        D = self.delays.numel()
        out = torch.zeros(T,B,Fch,D, device=x.device, dtype=x.dtype)
        for i,d in enumerate(self.delays.tolist()):
            if d==0: out[:,:,:,i]=x
            elif d<T: out[d:,:,:,i]=x[:T-d,:,:]
        return out


class ParallelRangeAzimuthSNN(nn.Module):
    def __init__(self, delays, freq_ch=8, hidden=128, beta=0.95):
        super().__init__()
        self.fb = FrequencyFilterBank(num_channels=freq_ch)
        self.range_delay = MultiChannelDelayBank(delays)
        self.az_delay = MultiChannelDelayBank(delays)

        feat_dim = freq_ch * len(delays)

        self.fc_r1 = nn.Linear(feat_dim, hidden)
        self.lif_r1 = snn.Leaky(beta=beta)
        self.fc_r2 = nn.Linear(hidden, RANGE_CLASSES)
        self.lif_r2 = snn.Leaky(beta=beta)

        self.fc_a1 = nn.Linear(feat_dim, hidden)
        self.lif_a1 = snn.Leaky(beta=beta)
        self.fc_a2 = nn.Linear(hidden, AZ_CLASSES)
        self.lif_a2 = snn.Leaky(beta=beta)

    def forward(self, tx, rx_l, rx_r):
        txf = self.fb(tx)
        rlf = self.fb(rx_l)
        rrf = self.fb(rx_r)

        rx_mid = 0.5 * (rlf + rrf)
        rx_diff = rrf - rlf

        dtx_r = self.range_delay(txf)
        dtx_a = self.az_delay(txf)

        feat_r = (dtx_r * rx_mid[..., None]).reshape(txf.shape[0], txf.shape[1], -1)
        feat_a = (dtx_a * rx_diff[..., None]).reshape(txf.shape[0], txf.shape[1], -1)

        self.lif_r1.reset_mem(); self.lif_r2.reset_mem()
        self.lif_a1.reset_mem(); self.lif_a2.reset_mem()
        mr1=mr2=ma1=ma2=None

        out_r=[]; out_a=[]
        for t in range(feat_r.shape[0]):
            sr1,mr1 = self.lif_r1(self.fc_r1(feat_r[t]), mr1)
            sr2,mr2 = self.lif_r2(self.fc_r2(sr1), mr2)
            sa1,ma1 = self.lif_a1(self.fc_a1(feat_a[t]), ma1)
            sa2,ma2 = self.lif_a2(self.fc_a2(sa1), ma2)
            out_r.append(sr2); out_a.append(sa2)

        return torch.stack(out_r,dim=0), torch.stack(out_a,dim=0)


def eval_model(model, loader, ce):
    model.eval()
    total_loss=0.0; cor_r=0; cor_a=0; tot=0
    yr_t=[]; yr_p=[]; ya_t=[]; ya_p=[]
    with torch.no_grad():
        for txb,rlb,rrb,yrb,yab in loader:
            txb=txb.to(DEVICE); rlb=rlb.to(DEVICE); rrb=rrb.to(DEVICE)
            yrb=yrb.to(DEVICE); yab=yab.to(DEVICE)
            tx_t=txb.unsqueeze(-1).permute(1,0,2)
            rl_t=rlb.unsqueeze(-1).permute(1,0,2)
            rr_t=rrb.unsqueeze(-1).permute(1,0,2)
            spk_r, spk_a = model(tx_t, rl_t, rr_t)
            cnt_r=spk_r.sum(dim=0); cnt_a=spk_a.sum(dim=0)
            loss = ce(cnt_r, yrb) + ce(cnt_a, yab)
            total_loss += loss.item() * yrb.size(0)
            pr = cnt_r.argmax(dim=1); pa = cnt_a.argmax(dim=1)
            cor_r += (pr == yrb).sum().item(); cor_a += (pa == yab).sum().item(); tot += yrb.size(0)
            yr_t.append(yrb.cpu().numpy()); yr_p.append(pr.cpu().numpy())
            ya_t.append(yab.cpu().numpy()); ya_p.append(pa.cpu().numpy())
    return total_loss/tot, cor_r/tot, cor_a/tot, np.concatenate(yr_t), np.concatenate(yr_p), np.concatenate(ya_t), np.concatenate(ya_p)


Xtx_tr,Xrl_tr,Xrr_tr,yr_tr,ya_tr,Xtx_te,Xrl_te,Xrr_te,yr_te,ya_te = build_data()
train_ds=TensorDataset(torch.from_numpy(Xtx_tr), torch.from_numpy(Xrl_tr), torch.from_numpy(Xrr_tr), torch.from_numpy(yr_tr), torch.from_numpy(ya_tr))
test_ds=TensorDataset(torch.from_numpy(Xtx_te), torch.from_numpy(Xrl_te), torch.from_numpy(Xrr_te), torch.from_numpy(yr_te), torch.from_numpy(ya_te))
train_loader=DataLoader(train_ds,batch_size=BATCH,shuffle=True)
test_loader=DataLoader(test_ds,batch_size=BATCH,shuffle=False)

model=ParallelRangeAzimuthSNN(DELAY_STEPS, freq_ch=FREQ_CH, hidden=HIDDEN, beta=0.95).to(DEVICE)
ce=nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(), lr=1e-3)

hist={"train_loss":[],"test_loss":[],"range_train_acc":[],"range_test_acc":[],"az_train_acc":[],"az_test_acc":[]}
best=None; best_test=float("inf")

for ep in range(1,EPOCHS+1):
    model.train(); run_loss=0.0; cor_r=0; cor_a=0; tot=0
    for txb,rlb,rrb,yrb,yab in train_loader:
        txb=txb.to(DEVICE); rlb=rlb.to(DEVICE); rrb=rrb.to(DEVICE)
        yrb=yrb.to(DEVICE); yab=yab.to(DEVICE)
        tx_t=txb.unsqueeze(-1).permute(1,0,2)
        rl_t=rlb.unsqueeze(-1).permute(1,0,2)
        rr_t=rrb.unsqueeze(-1).permute(1,0,2)
        opt.zero_grad()
        spk_r, spk_a = model(tx_t, rl_t, rr_t)
        cnt_r=spk_r.sum(dim=0); cnt_a=spk_a.sum(dim=0)
        loss = ce(cnt_r, yrb) + ce(cnt_a, yab)
        loss.backward(); opt.step()
        run_loss += loss.item()*yrb.size(0)
        pr = cnt_r.argmax(dim=1); pa = cnt_a.argmax(dim=1)
        cor_r += (pr==yrb).sum().item(); cor_a += (pa==yab).sum().item(); tot += yrb.size(0)

    tr_loss=run_loss/tot; tr_acc_r=cor_r/tot; tr_acc_a=cor_a/tot
    te_loss,te_acc_r,te_acc_a,yr_t,yr_p,ya_t,ya_p = eval_model(model,test_loader,ce)

    hist["train_loss"].append(tr_loss); hist["test_loss"].append(te_loss)
    hist["range_train_acc"].append(tr_acc_r); hist["range_test_acc"].append(te_acc_r)
    hist["az_train_acc"].append(tr_acc_a); hist["az_test_acc"].append(te_acc_a)

    if te_loss < best_test:
        best_test=te_loss; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    if ep==1 or ep%5==0 or ep==EPOCHS:
        print(f"Epoch {ep:02d}/{EPOCHS} | train_loss={tr_loss:.4f} | range_train={tr_acc_r*100:.1f}%, az_train={tr_acc_a*100:.1f}% | range_test={te_acc_r*100:.1f}%, az_test={te_acc_a*100:.1f}%")

model.load_state_dict(best)
torch.save({"model_state_dict":model.state_dict(),"history":hist}, CKPT)
print("Saved checkpoint:", CKPT)

# curves
epx=np.arange(1,EPOCHS+1)
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
ax[0].plot(epx,hist["train_loss"],label="train"); ax[0].plot(epx,hist["test_loss"],label="test"); ax[0].set_title("Loss"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].plot(epx,np.array(hist["range_train_acc"])*100,label="train"); ax[1].plot(epx,np.array(hist["range_test_acc"])*100,label="test"); ax[1].set_title("Range acc (%)"); ax[1].grid(alpha=0.3); ax[1].legend()
ax[2].plot(epx,np.array(hist["az_train_acc"])*100,label="train"); ax[2].plot(epx,np.array(hist["az_test_acc"])*100,label="test"); ax[2].set_title("Azimuth acc (%)"); ax[2].grid(alpha=0.3); ax[2].legend()
plt.tight_layout(); save_fig(PLOT_DIR,"curves_step5.png"); plt.show()

te_loss,te_acc_r,te_acc_a,yr_t,yr_p,ya_t,ya_p = eval_model(model,test_loader,ce)
confusion_matrix_plot(yr_t, yr_p, [f"{d:.1f}" for d in RANGE_CENTERS], PLOT_DIR, "confusion_range_step5.png")
confusion_matrix_plot(ya_t, ya_p, [f"{d:.0f}" for d in AZ_CENTERS], PLOT_DIR, "confusion_azimuth_step5.png")

with open(METRICS,"w",encoding="utf-8") as f:
    json.dump({"best_test_loss":best_test,"final_range_test_acc":float(te_acc_r),"final_az_test_acc":float(te_acc_a),"final_range_train_acc":float(hist["range_train_acc"][-1]),"final_az_train_acc":float(hist["az_train_acc"][-1])},f,indent=2)
print("Saved metrics:", METRICS)
